# Load data

In [151]:
import pandas as pd
import seaborn as sns
import matplotlib as plt
import matplotlib.pyplot as plt
import numpy as np
import warnings

warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

print("Loading data...")
# Master
df_pduct = pd.read_parquet('../dataset/cleaned/products.parquet')
df_cus = pd.read_parquet('../dataset/cleaned/customers.parquet')
df_pmot = pd.read_parquet('../dataset/cleaned/promotions.parquet')
df_geo = pd.read_parquet('../dataset/cleaned/geography.parquet')

# Transaction
df_ord = pd.read_parquet('../dataset/cleaned/orders.parquet')
df_item = pd.read_parquet('../dataset/cleaned/order_items.parquet')
df_ship = pd.read_parquet('../dataset/cleaned/shipments.parquet')
df_ret = pd.read_parquet('../dataset/cleaned/returns.parquet')
df_rev = pd.read_parquet('../dataset/cleaned/reviews.parquet')

# Analytical
df_sale = pd.read_parquet('../dataset/cleaned/sales.parquet')
#df_submit = pd.read_parquet('../dataset/sample_submission.csv')

# Operational
df_inv = pd.read_parquet('../dataset/cleaned/inventory.parquet')
df_web = pd.read_parquet('../dataset/cleaned/web_traffic.parquet')
print("Load data successfully")

Loading data...
Load data successfully


# Tạo feature và phân loại khách hàng

In [ ]:
total_refund_per_order = df_ret.groupby('order_id')['refund_amount'].sum()
df_ord['refund_amount'] = df_ord['order_id'].map(total_refund_per_order).fillna(0.0)
display(df_ord.sample(3))

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,payment_value,installments,refund_amount
533625,688108,2019-09-26,55971,30318,cancelled,cod,tablet,direct,8888.65,1 period,0.0
290552,374719,2016-04-06,22786,34451,delivered,paypal,mobile,organic_search,9046.38,6 periods,0.0
424345,547162,2017-11-26,153976,84050,delivered,paypal,desktop,paid_search,9276.55,6 periods,0.0


In [153]:
df_customer = df_cus.copy()

total_oder_per_cus = df_ord.groupby('customer_id')['order_id'].count()
total_return_oder_per_cus = df_ord[df_ord['order_status'] == 'returned'].groupby('customer_id')['order_id'].count()
total_cancelled_oder_per_cus = df_ord[df_ord['order_status'] == 'cancelled'].groupby('customer_id')['order_id'].count()
total_paid_per_cus = df_ord.groupby('customer_id')['payment_value'].sum()
total_refund_per_cus = df_ord.groupby('customer_id')['refund_amount'].sum()
total_refund_cancelled_per_cus = df_ord[df_ord['order_status'] == 'cancelled'].groupby('customer_id')['payment_value'].sum()

df_customer['total_oder'] = df_customer['customer_id'].map(total_oder_per_cus).fillna(0)
df_customer['return_oder'] = df_customer['customer_id'].map(total_return_oder_per_cus).fillna(0)
df_customer['cancel_oder'] = df_customer['customer_id'].map(total_cancelled_oder_per_cus).fillna(0)
df_customer['actual_oder'] = df_customer['total_oder'] - df_customer['return_oder'] - df_customer['cancel_oder']

df_customer['total_paid'] = df_customer['customer_id'].map(total_paid_per_cus).fillna(0)
df_customer['total_refund'] = df_customer['customer_id'].map(total_refund_per_cus).fillna(0)
df_customer['total_refund_cancel'] = df_customer['customer_id'].map(total_refund_cancelled_per_cus).fillna(0)
df_customer['actual_paid'] = df_customer['total_paid'] - df_customer['total_refund'] - df_customer['total_refund_cancel']

df_customer['return_cancel_rate'] = ((df_customer['return_oder'] + df_customer['cancel_oder']) / df_customer['total_oder']).fillna(-1.0)

display(df_customer.sample(3))

,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel,total_oder,return_oder,cancel_oder,actual_oder,total_paid,total_refund,total_refund_cancel,actual_paid,return_cancel_rate
42963,55372,31009,Thai Nguyen,2022-03-30,Female,35-44,social_media,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,-1.000000
19138,24678,45405,Bac Giang,2016-08-02,Female,18-24,organic_search,6.0,0.0,2.0,4.0,163427.50,0.00,69943.06,93484.44,0.333333
58419,75466,38119,Hoi An,2021-03-23,Male,25-34,organic_search,9.0,1.0,0.0,8.0,207129.74,5036.75,0.00,202092.99,0.111111


In [154]:
df_ord = df_ord.sort_values(by=['customer_id', 'order_date'])
df_ord['interval_prev_order'] = (
    df_ord.groupby('customer_id')['order_date']
    .diff()
    .dt.days
)
inter_order_gap = (
    df_ord.groupby('customer_id')['interval_prev_order']
    .apply(lambda x: x.iloc[1:].mean())
)
df_customer['avg_interval'] = df_customer['customer_id'].map(inter_order_gap).fillna(0.0)

first_order = df_ord.groupby('customer_id')['order_date'].first()
last_order = df_ord.groupby('customer_id')['order_date'].last()
df_customer['first_order_date'] = df_customer['customer_id'].map(first_order).fillna(df_customer['signup_date'])
df_customer['last_order_date'] = df_customer['customer_id'].map(last_order).fillna(df_customer['signup_date'])

current_date = pd.to_datetime('2022-12-31', dayfirst=True)
df_customer['recency_days'] = (current_date - df_customer['last_order_date']).dt.days
df_customer['tenure_days'] = (current_date - df_customer['first_order_date']).dt.days

df_customer = df_customer.drop(columns=['city', 'gender', 'age_group', 'acquisition_channel'])

In [155]:
df_customer.info()
display(df_customer.sample())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121930 entries, 0 to 121929
Data columns (total 17 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   customer_id          121930 non-null  string        
 1   zip                  121930 non-null  string        
 2   signup_date          121930 non-null  datetime64[ns]
 3   total_oder           121930 non-null  float64       
 4   return_oder          121930 non-null  float64       
 5   cancel_oder          121930 non-null  float64       
 6   actual_oder          121930 non-null  float64       
 7   total_paid           121930 non-null  float64       
 8   total_refund         121930 non-null  float64       
 9   total_refund_cancel  121930 non-null  float64       
 10  actual_paid          121930 non-null  float64       
 11  return_cancel_rate   121930 non-null  float64       
 12  avg_interval         121930 non-null  float64       
 13  first_order_da

,customer_id,zip,signup_date,total_oder,return_oder,cancel_oder,actual_oder,total_paid,total_refund,total_refund_cancel,actual_paid,return_cancel_rate,avg_interval,first_order_date,last_order_date,recency_days,tenure_days
58054,74993,38553,2019-04-04,1.0,0.0,0.0,1.0,45924.9,0.0,0.0,45924.9,0.0,0.0,2014-05-17,2014-05-17,3150,3150


In [156]:
six_months_ago = pd.to_datetime('2022-06-01', dayfirst=True)

conditions = [
    # 1. Non-buyer
    (df_customer['total_oder'] == 0),

    # 2. Bad quality
    (df_customer['return_cancel_rate'] >= 0.5),

    # 3. Lost
    (df_customer['recency_days'] > 730),

    # 4. New customer
    (df_customer['first_order_date'] >= six_months_ago),

    # 5. One-time buyer
    (df_customer['actual_oder'] == 1),

    # 6. At-risk
    (df_customer['actual_oder'] > 1) &
    (df_customer['recency_days'] > df_customer['avg_interval'] * 1.5),
]

choices = [
    'non_buyer',
    'bad_customer',
    'lost',
    'new_customer',
    'one_time',
    'at_risk'
]

df_customer['customer_segment'] = np.select(conditions, choices, default='loyal')

In [160]:
display(df_customer.sample(5))

,customer_id,zip,signup_date,total_oder,return_oder,cancel_oder,actual_oder,total_paid,total_refund,total_refund_cancel,actual_paid,return_cancel_rate,avg_interval,first_order_date,last_order_date,recency_days,tenure_days,customer_segment
92180,119106,60614,2019-05-07,10.0,2.0,1.0,7.0,109899.08,29424.15,2313.74,78161.19,0.300000,324.333333,2013-08-03,2021-07-31,518,3437,at_risk
40088,51648,28386,2018-06-02,4.0,0.0,1.0,3.0,82595.23,0.00,30838.27,51756.96,0.250000,630.000000,2017-04-29,2022-07-02,182,2072,loyal
115285,148953,98178,2015-09-29,6.0,0.0,1.0,5.0,114153.18,0.00,10536.24,103616.94,0.166667,736.600000,2012-11-11,2022-12-12,19,3702,loyal
11067,14277,12569,2018-07-16,13.0,1.0,1.0,11.0,314575.63,3940.00,2433.27,308202.36,0.153846,237.500000,2013-08-29,2021-06-18,561,3411,at_risk
18059,23317,32439,2020-05-26,4.0,0.0,0.0,4.0,91780.48,0.00,0.00,91780.48,0.000000,681.666667,2015-03-10,2020-10-14,808,2853,lost
